# Deteksi Anomali Keuangan dengan Temporal dan Network Feature Engineering

Menggunakan fitur temporal dan jaringan berdasarkan kolom transaksi yang tersedia pada IBM AML dan PaySim. Kolom `amount_gradient` dan `amount_pct_change` mengukur perubahan nominal transaksi terhadap rata-rata nominal pada timestamp sebelumnya untuk pengirim yang sama.

Fitur `source_strength_prior` dan `destination_strength_prior` merupakan jumlah nominal historis keluar/ masuk sebelum timestamp terbaru. `new_counterparty` menandai pasangan sumber-tujuan yang belum memiliki transaksi terdahulu.

## 1. Install packages
DuckDB dipakai agar fitur historis dihitung dari **seluruh file transaksi** secara out-of-core sebelum mengambil sampel untuk eksperimen.

In [ ]:
%pip -q install kagglehub pandas numpy scikit-learn matplotlib duckdb

In [ ]:
import os, glob, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import duckdb
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")
SEED = 42
pd.set_option("display.max_columns", 100)
print("Ready.")

## 2. Download datasets from Kaggle

In [ ]:
ibm_path = kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
print("Path to IBM AML dataset files:", ibm_path)

paysim_path = kagglehub.dataset_download("ealaxi/paysim1")
print("Path to PaySim dataset files:", paysim_path)

## 3. Inventory CSV files

In [ ]:
def list_csv_files(root):
    rows = []
    for p in Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() == ".csv":
            rows.append({"path": str(p), "file": p.name,
                         "size_mb": round(p.stat().st_size / 1024**2, 2)})
    return pd.DataFrame(rows).sort_values("size_mb", ascending=False).reset_index(drop=True)

ibm_files = list_csv_files(ibm_path)
paysim_files = list_csv_files(paysim_path)
print("IBM AML files"); display(ibm_files)
print("PaySim files"); display(paysim_files)

## 4. Select files and preview
Change either path if the first file listed is not the transaction table.

In [ ]:
if ibm_files.empty or paysim_files.empty:
    raise FileNotFoundError("No CSV files found")

small_candidates = ibm_files[ibm_files["file"].eq("HI-Small_Trans.csv")]
if small_candidates.empty:
    raise FileNotFoundError("HI-Small_Trans.csv tidak ditemukan. Pilih file transaksi IBM AML yang lebih kecil secara manual")
IBM_CSV_PATH = small_candidates.iloc[0]["path"]
PAYSIM_CSV_PATH = paysim_files.iloc[0]["path"]

print("IBM selected (full history):", IBM_CSV_PATH)
print("PaySim selected (full history):", PAYSIM_CSV_PATH)

# Preview
ibm_preview = pd.read_csv(IBM_CSV_PATH, nrows=5, low_memory=True)
paysim_preview = pd.read_csv(PAYSIM_CSV_PATH, nrows=5, low_memory=True)
display(ibm_preview)
display(paysim_preview)

## 5. Audit skema ringan

Read 1.000 baris untuk mengenali nama kolom. Fitur historis akan dihitung dengan DuckDB pada seluruh file transaksi, kemudian dipilih sampel untuk melatih dan mengevaluasi model.

In [ ]:
SCHEMA_PROBE_ROWS = 1_000
ibm_raw = pd.read_csv(IBM_CSV_PATH, nrows=SCHEMA_PROBE_ROWS, low_memory=True)
paysim_raw = pd.read_csv(PAYSIM_CSV_PATH, nrows=SCHEMA_PROBE_ROWS, low_memory=True)

def schema_audit(df, name):
    audit = pd.DataFrame({
        "column": df.columns.astype(str),
        "dtype": [str(df[c].dtype) for c in df.columns],
        "missing_in_probe": [int(df[c].isna().sum()) for c in df.columns],
        "unique_in_probe": [int(df[c].nunique()) for c in df.columns]
    })
    print(f"{name}: audit probe {len(df):,} baris (bukan sampel model)")
    display(audit)
    display(df.head())
    return audit

ibm_audit = schema_audit(ibm_raw, "IBM AML")
paysim_audit = schema_audit(paysim_raw, "PaySim")

## 6. Map columns
The automatic mapping uses common column names.

In [ ]:
def find_col(columns, candidates, required=False, field=""):
    cols = list(columns)
    lower = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    norm = lambda x: "".join(ch for ch in str(x).lower() if ch.isalnum())
    normalized = {norm(c): c for c in cols}
    for cand in candidates:
        if norm(cand) in normalized:
            return normalized[norm(cand)]
    if required:
        raise KeyError(f"Cannot map required field {field}. Available columns: {cols}. Set manually.")
    return None

IBM_COLS = {
    "timestamp": find_col(ibm_raw.columns, ["Timestamp", "tran_timestamp", "timestamp"], True, "timestamp"),
    "source_account": find_col(ibm_raw.columns, ["Account", "orig_acct", "Origin Account", "Source Account"], True, "source_account"),
    "destination_account": find_col(ibm_raw.columns, ["Account.1", "bene_acct", "Destination Account", "Beneficiary Account"], True, "destination_account"),
    "source_bank": find_col(ibm_raw.columns, ["From Bank", "from_bank", "Origin Bank", "source_bank"]),
    "destination_bank": find_col(ibm_raw.columns, ["To Bank", "to_bank", "Beneficiary Bank", "destination_bank"]),
    "amount": find_col(ibm_raw.columns, ["Amount Received", "base_amt", "amount", "Amount Paid", "Amount"], True, "amount"),
    "label": find_col(ibm_raw.columns, ["Is Laundering", "is_sar", "is_laundering", "label", "isFraud"])
}
PAYSIM_COLS = {
    "timestamp": find_col(paysim_raw.columns, ["step", "timestamp", "trans_date_trans_time"], True, "timestamp"),
    "source_account": find_col(paysim_raw.columns, ["nameOrig", "source_account", "orig_acct"], True, "source_account"),
    "destination_account": find_col(paysim_raw.columns, ["nameDest", "destination_account", "bene_acct"], True, "destination_account"),
    "source_bank": None, "destination_bank": None,
    "amount": find_col(paysim_raw.columns, ["amount", "Amount"], True, "amount"),
    "label": find_col(paysim_raw.columns, ["isFraud", "is_fraud", "label"])
}
print("IBM mapping:\n", json.dumps(IBM_COLS, indent=2, default=str))
print("PaySim mapping:\n", json.dumps(PAYSIM_COLS, indent=2, default=str))

## 7. Normalize transactions

Pada PaySim, `step` merupakan **indeks waktu simulasi**. Timestamp ini hanya untuk mengurutkan transaksi dan split kronologis, serta menyimpan nilai asli pada `time_index`. Fitur saldo dan flag fraud tidak digunakan sebagai model utama karena dapat membocorkan mekanisme simulasi.

In [ ]:
def normalize_transactions(raw, cols, dataset_name):
    out = pd.DataFrame(index=raw.index)
    raw_time = raw[cols["timestamp"]]
    if dataset_name.lower() == "paysim" and pd.api.types.is_numeric_dtype(raw_time):
        step = pd.to_numeric(raw_time, errors="coerce")
        out["time_index"] = step.astype(float)
        # Internal ordering only; this is NOT a real calendar timestamp.
        out["timestamp"] = pd.to_datetime(step, unit="h", origin="unix", errors="coerce")
    else:
        out["timestamp"] = pd.to_datetime(raw_time, errors="coerce", utc=True)
    out["source_account"] = raw[cols["source_account"]].astype("string")
    out["destination_account"] = raw[cols["destination_account"]].astype("string")
    out["amount"] = pd.to_numeric(raw[cols["amount"]], errors="coerce")
    if cols.get("source_bank") is not None:
        out["source_bank"] = raw[cols["source_bank"]].astype("string")
    if cols.get("destination_bank") is not None:
        out["destination_bank"] = raw[cols["destination_bank"]].astype("string")
    if cols.get("label") is not None:
        out["label"] = pd.to_numeric(raw[cols["label"]], errors="coerce")
    out["source_row_id"] = np.arange(len(out))
    out = out.dropna(subset=["timestamp", "source_account", "destination_account", "amount"]).copy()
    out["amount_log"] = np.log1p(out["amount"].clip(lower=0))
    out = out.sort_values(["timestamp", "source_row_id"], kind="mergesort").reset_index(drop=True)
    out["row_id"] = np.arange(len(out))
    return out

ibm_tx = normalize_transactions(ibm_raw, IBM_COLS, "ibm")
paysim_tx = normalize_transactions(paysim_raw, PAYSIM_COLS, "paysim")
print("IBM:", ibm_tx.shape, ibm_tx.timestamp.min(), ibm_tx.timestamp.max())
print("PaySim:", paysim_tx.shape, "simulation step range:", paysim_tx.time_index.min(), "to", paysim_tx.time_index.max())
display(ibm_tx.head())

## 8. Leakage-safe historical features

Histori dihitung menggunakan seluruh transaksi sumber sebelum pengambilan sampel. Fitur prior-count hanya menghitung timestamp yang lebih awal. Pada PaySim, `step` tetap sebagai indeks waktu simulasi (1 step = 1 jam simulasi), bukan kalender.

In [ ]:
import duckdb
import os

DUCKDB_MEMORY_LIMIT = "4GB"
DUCKDB_THREADS = 2
DUCKDB_TEMP_DIR = "/content/duckdb_fraud_tmp"
os.makedirs(DUCKDB_TEMP_DIR, exist_ok=True)


def qi(name):
    """Quote nama kolom/tabel SQL dengan aman"""
    return '"' + str(name).replace('"', '""') + '"'


def sql_path(path):
    """Escape path untuk literal string SQL"""
    return "'" + str(path).replace("'", "''") + "'"


def normalize_col(value):
    """Normalisasi nama kolom untuk pencocokan"""
    return "".join(
        ch.lower() for ch in str(value) if ch.isalnum()
    )


def resolve_duckdb_columns(con, csv_path, cols):
    """Petakan nama kolom konfigurasi ke nama aktual di DuckDB"""
    desc = con.execute(
        f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto(
            {sql_path(csv_path)},
            header=true,
            all_varchar=true,
            sample_size=100000,
            ignore_errors=true
        )
        """
    ).fetchall()

    duck_columns = [row[0] for row in desc]

    # Exact match lebih diutamakan.
    exact = {}
    normalized = {}

    for col in duck_columns:
        exact.setdefault(str(col).strip().lower(), col)
        normalized.setdefault(normalize_col(col), col)

    resolved = {}

    for field, value in cols.items():
        if value is None:
            resolved[field] = None
            continue

        key = str(value).strip().lower()

        if key in exact:
            resolved[field] = exact[key]
        elif normalize_col(value) in normalized:
            resolved[field] = normalized[normalize_col(value)]
        else:
            raise KeyError(
                f"Kolom untuk '{field}' tidak ditemukan: {value!r}\n"
                f"Kolom yang tersedia: {duck_columns}"
            )

    print("Kolom yang dipetakan:", resolved)
    return resolved


def build_full_history_sample(
    csv_path, cols, dataset_name, sample_size, seed=42
):
    con = duckdb.connect()
    try:
        con.execute(
            f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'"
        )
        con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
        con.execute(
            f"SET temp_directory={sql_path(DUCKDB_TEMP_DIR)}"
        )

        cols = resolve_duckdb_columns(con, csv_path, cols)

        prefix = (
            "ibm" if dataset_name.lower() == "ibm" else "paysim"
        )
        base_table = f"{prefix}_base"
        feat_table = f"{prefix}_features"
        sample_table = f"{prefix}_sample"

        ts_col = qi(cols["timestamp"])
        src_col = qi(cols["source_account"])
        dst_col = qi(cols["destination_account"])
        amt_col = qi(cols["amount"])
        label_col = (
            qi(cols["label"]) if cols.get("label") else None
        )
        source_bank_col = (
            qi(cols["source_bank"])
            if cols.get("source_bank") else None
        )
        destination_bank_col = (
            qi(cols["destination_bank"])
            if cols.get("destination_bank") else None
        )

        is_paysim = dataset_name.lower() == "paysim"

        if is_paysim:
            timestamp_expr = (
                "(TIMESTAMP '1970-01-01 00:00:00' + "
                f"TRY_CAST({ts_col} AS BIGINT) * INTERVAL '1 hour')"
            )
            time_index_select = (
                f", TRY_CAST({ts_col} AS DOUBLE) AS time_index"
            )
        else:
            timestamp_expr = (
                f"TRY_STRPTIME(CAST({ts_col} AS VARCHAR), "
                "'%Y/%m/%d %H:%M')"
            )
            time_index_select = ""

        bank_select = ""
        if source_bank_col and destination_bank_col:
            bank_select = (
                f", CAST({source_bank_col} AS VARCHAR) AS source_bank"
                f", CAST({destination_bank_col} AS VARCHAR) AS destination_bank"
            )

        label_select = (
            f", TRY_CAST({label_col} AS INTEGER) AS label"
            if label_col else ""
        )

        con.execute(f"DROP TABLE IF EXISTS {base_table}")
        con.execute(
            f"""
            CREATE TABLE {base_table} AS
            SELECT
                ROW_NUMBER() OVER (
                    ORDER BY
                        {timestamp_expr},
                        CAST({src_col} AS VARCHAR),
                        CAST({dst_col} AS VARCHAR),
                        TRY_CAST({amt_col} AS DOUBLE)
                        {', CAST(' + source_bank_col + ' AS VARCHAR)' if source_bank_col else ''}
                        {', CAST(' + destination_bank_col + ' AS VARCHAR)' if destination_bank_col else ''}
                ) AS row_id,
                {timestamp_expr} AS timestamp
                {time_index_select},
                CAST({src_col} AS VARCHAR) AS source_account,
                CAST({dst_col} AS VARCHAR) AS destination_account,
                TRY_CAST({amt_col} AS DOUBLE) AS amount
                {bank_select}
                {label_select}
            FROM read_csv_auto(
                {sql_path(csv_path)},
                header=true,
                all_varchar=true,
                sample_size=100000,
                ignore_errors=true
            )
            WHERE {timestamp_expr} IS NOT NULL
              AND {src_col} IS NOT NULL
              AND {dst_col} IS NOT NULL
              AND TRY_CAST({amt_col} AS DOUBLE) IS NOT NULL
            """
        )

        n_full = con.execute(
            f"SELECT COUNT(*) FROM {base_table}"
        ).fetchone()[0]
        print(
            f"{dataset_name}: transaksi valid = {n_full:,}"
        )

        # Hitung histori berdasarkan waktu
        entities = [
            (
                "source",
                "source_account",
                "source_prior_count",
                "source_seconds_since_previous",
            ),
            (
                "destination",
                "destination_account",
                "destination_prior_count",
                "destination_seconds_since_previous",
            ),
            (
                "account_pair",
                "source_account, destination_account",
                "account_pair_prior_count",
                None,
            ),
        ]

        for entity, keys, prior_name, gap_name in entities:
            hist = f"{prefix}_{entity}_hist"
            gap_sql = (
                ", DATE_DIFF('second', previous_timestamp, timestamp)"
                " AS seconds_since_previous"
                if gap_name else ""
            )

            con.execute(f"DROP TABLE IF EXISTS {hist}")
            con.execute(
                f"""
                CREATE TABLE {hist} AS
                WITH counts AS (
                    SELECT {keys}, timestamp, COUNT(*) AS n_at_time
                    FROM {base_table}
                    GROUP BY {keys}, timestamp
                ),
                windows AS (
                    SELECT *,
                        SUM(n_at_time) OVER (
                            PARTITION BY {keys}
                            ORDER BY timestamp
                            ROWS BETWEEN UNBOUNDED PRECEDING
                                     AND CURRENT ROW
                        ) - n_at_time AS prior_count,
                        LAG(timestamp) OVER (
                            PARTITION BY {keys}
                            ORDER BY timestamp
                        ) AS previous_timestamp
                    FROM counts
                )
                SELECT {keys}, timestamp, prior_count
                    {gap_sql}
                FROM windows
                """
            )

        source_amount_hist = f"{prefix}_source_amount_hist"
        con.execute(f"DROP TABLE IF EXISTS {source_amount_hist}")
        con.execute(f"""
            CREATE TABLE {source_amount_hist} AS
            WITH by_time AS (
                SELECT source_account, timestamp,
                       COUNT(*) AS n_at_time,
                       SUM(amount) AS amount_sum_at_time,
                       AVG(amount) AS amount_mean_at_time
                FROM {base_table}
                GROUP BY source_account, timestamp
            ), win AS (
                SELECT *,
                    SUM(n_at_time) OVER (
                        PARTITION BY source_account ORDER BY timestamp
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                    ) - n_at_time AS prior_amount_count,
                    SUM(amount_sum_at_time) OVER (
                        PARTITION BY source_account ORDER BY timestamp
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                    ) - amount_sum_at_time AS prior_amount_sum,
                    LAG(amount_mean_at_time) OVER (
                        PARTITION BY source_account ORDER BY timestamp
                    ) AS previous_amount_mean
                FROM by_time
            )
            SELECT * FROM win
        """)

        # Network strength
        source_strength_hist = f"{prefix}_source_strength_hist"
        con.execute(f"DROP TABLE IF EXISTS {source_strength_hist}")
        con.execute(f"""
            CREATE TABLE {source_strength_hist} AS
            WITH by_time AS (
                SELECT source_account, timestamp, SUM(amount) AS amount_at_time
                FROM {base_table} GROUP BY source_account, timestamp
            )
            SELECT source_account, timestamp,
                SUM(amount_at_time) OVER (
                    PARTITION BY source_account ORDER BY timestamp
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) - amount_at_time AS source_strength_prior
            FROM by_time
        """)

        destination_strength_hist = f"{prefix}_destination_strength_hist"
        con.execute(f"DROP TABLE IF EXISTS {destination_strength_hist}")
        con.execute(f"""
            CREATE TABLE {destination_strength_hist} AS
            WITH by_time AS (
                SELECT destination_account, timestamp, SUM(amount) AS amount_at_time
                FROM {base_table} GROUP BY destination_account, timestamp
            )
            SELECT destination_account, timestamp,
                SUM(amount_at_time) OVER (
                    PARTITION BY destination_account ORDER BY timestamp
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) - amount_at_time AS destination_strength_prior
            FROM by_time
        """)

        # Histori pasangan antarbank, hanya jika kolom tersedia
        bank_pair_available = bool(
            source_bank_col and destination_bank_col
        )

        if bank_pair_available:
            hist = f"{prefix}_bank_pair_hist"
            con.execute(f"DROP TABLE IF EXISTS {hist}")
            con.execute(
                f"""
                CREATE TABLE {hist} AS
                WITH counts AS (
                    SELECT source_bank, destination_bank,
                           timestamp, COUNT(*) AS n_at_time
                    FROM {base_table}
                    GROUP BY source_bank, destination_bank, timestamp
                )
                SELECT source_bank, destination_bank, timestamp,
                    SUM(n_at_time) OVER (
                        PARTITION BY source_bank, destination_bank
                        ORDER BY timestamp
                        ROWS BETWEEN UNBOUNDED PRECEDING
                                 AND CURRENT ROW
                    ) - n_at_time AS bank_pair_prior_count
                FROM counts
                """
            )

        joins = f"""
            LEFT JOIN {prefix}_source_hist s
              ON b.source_account = s.source_account
             AND b.timestamp = s.timestamp
            LEFT JOIN {prefix}_destination_hist d
              ON b.destination_account = d.destination_account
             AND b.timestamp = d.timestamp
            LEFT JOIN {prefix}_account_pair_hist p
              ON b.source_account = p.source_account
             AND b.destination_account = p.destination_account
             AND b.timestamp = p.timestamp
            LEFT JOIN {prefix}_source_amount_hist sa
              ON b.source_account = sa.source_account
             AND b.timestamp = sa.timestamp
            LEFT JOIN {prefix}_source_strength_hist ss
              ON b.source_account = ss.source_account
             AND b.timestamp = ss.timestamp
            LEFT JOIN {prefix}_destination_strength_hist ds
              ON b.destination_account = ds.destination_account
             AND b.timestamp = ds.timestamp
        """

        extra_select = """
            , s.prior_count AS source_prior_count
            , s.seconds_since_previous
                AS source_seconds_since_previous
            , d.prior_count AS destination_prior_count
            , d.seconds_since_previous
                AS destination_seconds_since_previous
            , p.prior_count AS account_pair_prior_count
            , sa.prior_amount_sum AS source_amount_sum_prior
            , sa.previous_amount_mean AS source_previous_amount_mean
            , CASE WHEN sa.prior_amount_count > 0
                   THEN sa.prior_amount_sum / NULLIF(sa.prior_amount_count, 0)
                   ELSE NULL END AS source_amount_mean_prior
            , CASE WHEN sa.previous_amount_mean IS NOT NULL
                   THEN b.amount - sa.previous_amount_mean ELSE NULL END AS amount_gradient
            , CASE WHEN sa.previous_amount_mean IS NOT NULL
                         AND ABS(sa.previous_amount_mean) > 0
                   THEN (b.amount - sa.previous_amount_mean) / ABS(sa.previous_amount_mean)
                   ELSE NULL END AS amount_pct_change
            , CASE WHEN s.seconds_since_previous > 0
                   THEN 3600.0 / s.seconds_since_previous ELSE NULL END AS source_velocity_per_hour
            , ss.source_strength_prior
            , ds.destination_strength_prior
            , (COALESCE(ds.destination_strength_prior, 0) + 1.0)
                / (COALESCE(ss.source_strength_prior, 0) + 1.0) AS in_out_strength_ratio
            , CASE WHEN COALESCE(p.prior_count, 0) = 0 THEN 1 ELSE 0 END
                AS new_counterparty
        """

        if bank_pair_available:
            joins += f"""
                LEFT JOIN {prefix}_bank_pair_hist bp
                  ON b.source_bank = bp.source_bank
                 AND b.destination_bank = bp.destination_bank
                 AND b.timestamp = bp.timestamp
            """
            extra_select += """
                , bp.bank_pair_prior_count
                , CASE
                    WHEN b.source_bank IS NOT NULL
                     AND b.destination_bank IS NOT NULL
                     AND b.source_bank <> b.destination_bank
                    THEN 1 ELSE 0
                  END AS is_interbank_pair
            """

        calendar_select = ""
        if not is_paysim:
            calendar_select = """
                , EXTRACT(HOUR FROM b.timestamp)::DOUBLE AS hour
                , (EXTRACT(ISODOW FROM b.timestamp) - 1)::DOUBLE
                    AS day_of_week
            """

        con.execute(f"DROP TABLE IF EXISTS {feat_table}")
        con.execute(
            f"""
            CREATE TABLE {feat_table} AS
            SELECT
                b.*,
                LN(1 + GREATEST(b.amount, 0)) AS amount_log
                {extra_select}
                {calendar_select}
            FROM {base_table} b
            {joins}
            """
        )

        # row_id stabil dengan mengurutkan baris sumber
        con.execute(f"DROP TABLE IF EXISTS {sample_table}")
        con.execute(
            f"""
            CREATE TABLE {sample_table} AS
            SELECT *
            FROM {feat_table}
            ORDER BY
                MD5(CAST(row_id AS VARCHAR) || ':' || CAST({int(seed)} AS VARCHAR)),
                row_id
            LIMIT {int(sample_size)}
            """
        )

        result = con.execute(
            f"""
            SELECT *
            FROM {sample_table}
            ORDER BY timestamp, row_id
            """
        ).df()

        print(
            f"{dataset_name}: sampel setelah full-history = "
            f"{len(result):,} baris"
        )
        return result

    finally:
        con.close()


ibm_feat = build_full_history_sample(
    IBM_CSV_PATH, IBM_COLS, "IBM", 100_000, SEED
)
paysim_feat = build_full_history_sample(
    PAYSIM_CSV_PATH, PAYSIM_COLS, "PaySim", 50_000, SEED + 1
)

print("IBM feature shape:", ibm_feat.shape)
print("PaySim feature shape:", paysim_feat.shape)

display(ibm_feat.head())
display(paysim_feat.head())


## 9. Chronological train/test split

Train-test dibagi berdasarkan **unique timestamp** agar tidak terpecah ke dua set. Batas dipilih dari titik waktu yang membuat jumlah baris train paling dekat dengan 70%. Pada PaySim, timestamp merepresentasikan urutan `step` simulasi, bukan kalender nyata.

In [ ]:
def chronological_split(feat, train_fraction=0.70):
    clean = feat.dropna(subset=["timestamp"]).copy()
    counts = clean.groupby("timestamp", sort=True).size()
    if len(counts) < 2:
        raise ValueError("Need at least two unique timestamps for chronological split")
    # Candidate cut positions
    cumulative = counts.cumsum().iloc[:-1]
    target = train_fraction * len(clean)
    best_pos = int(np.argmin(np.abs(cumulative.to_numpy() - target)))
    cut = counts.index[best_pos + 1]  # first timestamp assigned to test
    train = clean[clean.timestamp < cut].copy()
    test = clean[clean.timestamp >= cut].copy()
    if train.empty or test.empty:
        raise ValueError("Empty train/test split.")
    assert train.timestamp.max() < test.timestamp.min()
    return train, test, cut


def report_split(name, train, test, cut):
    total = len(train) + len(test)
    print(f"{name}: cutoff={cut}; train={len(train):,} ({len(train)/total:.1%}); "
          f"test={len(test):,} ({len(test)/total:.1%})")
    if "label" in train.columns:
        for part_name, part in [("train", train), ("test", test)]:
            y = pd.to_numeric(part["label"], errors="coerce")
            print(f"  {part_name} label counts: {y.value_counts(dropna=False).to_dict()}")

ibm_train, ibm_test, ibm_cut = chronological_split(ibm_feat)
paysim_train, paysim_test, paysim_cut = chronological_split(paysim_feat)
report_split("IBM AML", ibm_train, ibm_test, ibm_cut)
report_split("PaySim", paysim_train, paysim_test, paysim_cut)

# Explicit evaluation-readiness check
def check_test_labels(name, test):
    if "label" not in test.columns:
        print(f"WARNING {name}: no label mapped; supervised metrics unavailable.")
        return
    y = pd.to_numeric(test["label"], errors="coerce").dropna().astype(int)
    counts = y.value_counts().to_dict()
    if len(y.unique()) < 2:
        print(f"WARNING {name}: test set has only one label class: {counts} "
              "Increase sample size or verify label mapping; do not interpret ROC-AUC/PR-AUC")
    elif int((y == 1).sum()) == 0:
        print(f"WARNING {name}: no positive labels in test set. Increase sample size / verify mapping")
    else:
        print(f"{name}: evaluation has both classes; positive test cases={(y == 1).sum():,}")

check_test_labels("IBM AML", ibm_test)
check_test_labels("PaySim", paysim_test)

## 10. Define experiments

* E1 baseline
* E2 menambahkan fitur historis temporal
* E3 menambahkan fitur akun/jaringan
* E4 menggabungkan keduanya.

Pada PaySim, fitur kalender `hour` dan `day_of_week` tidak digunakan karena `step` bukan waktu kalender. Fitur `source_seconds_since_previous` merepresentasikan selisih `step` dalam jam simulasi.

In [ ]:
def feature_sets(feat):
    """Explicit feature whitelist. Labels, IDs and raw timestamps are never features"""
    baseline = ["amount_log"]
    temporal = [
        "source_prior_count",
        "source_seconds_since_previous",
        "source_amount_mean_prior",
        "amount_gradient",
        "amount_pct_change",
        "source_velocity_per_hour",
    ]
    if "time_index" not in feat.columns:
        temporal += ["hour", "day_of_week"]
    network = [
        "destination_prior_count",
        "account_pair_prior_count",
        "source_strength_prior",
        "destination_strength_prior",
        "in_out_strength_ratio",
        "new_counterparty",
    ]
    if "bank_pair_prior_count" in feat.columns:
        network.append("bank_pair_prior_count")
    if "is_interbank_pair" in feat.columns:
        network.append("is_interbank_pair")
    sets = {
        "E1_Baseline": baseline,
        "E2_Temporal": baseline + temporal,
        "E3_Network": baseline + network,
        "E4_Fusion": baseline + temporal + network,
    }
    for name, cols in sets.items():
        missing = [c for c in cols if c not in feat.columns]
        if missing:
            raise KeyError(f"{name} missing {missing}; periksa cell fitur historis sebelum training")
    return sets

IBM_FEATURE_SETS = feature_sets(ibm_feat)
PAYSIM_FEATURE_SETS = feature_sets(paysim_feat)
print("IBM", IBM_FEATURE_SETS)
print("PaySim", PAYSIM_FEATURE_SETS)

In [ ]:
BLOCKED_FEATURE_TOKENS = {"label", "is_fraud", "isfraud", "fraud", "is_laundering", "is_sar",
                          "timestamp", "row_id", "source_row_id", "source_account", "destination_account",
                          "source_bank", "destination_bank", "time_index", "amount", "isflaggedfraud"}
for ds_name, frame in [("IBM_AML", ibm_feat), ("PaySim", paysim_feat)]:
    fsets = feature_sets(frame)
    for exp_name, cols in fsets.items():
        bad = [c for c in cols if str(c).lower() in BLOCKED_FEATURE_TOKENS]
        if bad:
            raise AssertionError(f"Potential leakage in {ds_name}/{exp_name}: {bad}")
        nonnumeric = [c for c in cols if not pd.api.types.is_numeric_dtype(frame[c])]
        if nonnumeric:
            raise AssertionError(f"Non-numeric features in {ds_name}/{exp_name}: {nonnumeric}")
    print(f"{ds_name} feature audit:")
    for exp_name, cols in fsets.items():
        print(f"  {exp_name}: {len(cols)} requested features -> {cols}")
print("Feature audit passed. Historical aggregates are constructed before sampling and exclude same-timestamp transactions")


## 11. Train Isolation Forest and evaluate

Skor yang lebih tinggi berarti lebih anomali (`-model.decision_function`). `PR_AUC` dihitung dengan `average_precision_score` (Average Precision/AP). Jumlah kasus positif test dicatat karena fraud sangat jarang. ROC rendah kemungkinan fitur/model merangking anomali secara buruk.


In [ ]:

def precision_recall_at_k(y, scores, k):
    """Hitung precision dan recall pada top-k skor anomali tertinggi"""
    y = np.asarray(y).astype(int)
    scores = np.asarray(scores, dtype=float)

    if len(y) == 0 or len(scores) == 0:
        return np.nan, np.nan

    k = min(max(int(k), 1), len(scores))
    top_idx = np.argsort(scores)[::-1][:k]

    precision = float(y[top_idx].mean())
    total_positive = int(y.sum())
    recall = (
        float(y[top_idx].sum() / total_positive)
        if total_positive > 0
        else np.nan
    )

    return precision, recall


def run_experiments(train, test, sets, dataset_name):
    """
    Jalankan E1-E4 menggunakan Isolation Forest
    Skor lebih tinggi selalu menunjukkan transaksi lebih anomali
    """
    rows = []

    metadata_cols = [
        c for c in [
            "row_id", "timestamp", "source_account",
            "destination_account", "amount", "label"
        ]
        if c in test.columns
    ]
    score_table = test[metadata_cols].copy()
    fitted = {}

    for name, feature_cols in sets.items():
        print(f"Menjalankan {dataset_name} - {name}...")

        # Validasi fitur
        missing_cols = [
            c for c in feature_cols
            if c not in train.columns or c not in test.columns
        ]
        if missing_cols:
            raise KeyError(
                f"Fitur {missing_cols} tidak ditemukan untuk {name}"
            )

        X_train = train[feature_cols].copy()
        X_test = test[feature_cols].copy()

        X_train = X_train.apply(pd.to_numeric, errors="coerce")
        X_test = X_test.apply(pd.to_numeric, errors="coerce")

        X_train = X_train.replace([np.inf, -np.inf], np.nan)
        X_test = X_test.replace([np.inf, -np.inf], np.nan)

        usable_cols = [
            c for c in feature_cols
            if X_train[c].notna().any()
        ]

        if not usable_cols:
            raise ValueError(
                f"Tidak ada fitur valid pada eksperimen {name}"
            )

        X_train = X_train[usable_cols]
        X_test = X_test[usable_cols]

        imputer = SimpleImputer(strategy="median")
        X_train_imp = imputer.fit_transform(X_train)
        X_test_imp = imputer.transform(X_test)

        model = IsolationForest(
            n_estimators=200,
            max_samples="auto",
            contamination="auto",
            random_state=SEED,
            n_jobs=-1
        )
        model.fit(X_train_imp)

        # Isolation Forest memiliki decision_function lebih rendah
        anomaly_scores = -model.decision_function(X_test_imp)

        row = {
            "dataset": dataset_name,
            "experiment": name,
            "n_features": len(usable_cols),
            "n_train": len(X_train_imp),
            "n_test": len(X_test_imp),
            "PR_AUC": np.nan,
            "ROC_AUC": np.nan,
            "n_positive_test": 0,
            "positive_rate_test": np.nan,
            "Precision_at_1pct": np.nan,
            "Recall_at_1pct": np.nan
        }

        # Label hanya dipakai untuk evaluasi, bukan sebagai fitur
        if "label" in test.columns:
            labels = pd.to_numeric(
                test["label"], errors="coerce"
            )
            valid = labels.notna()

            y = labels.loc[valid].astype(int).to_numpy()
            scores_valid = anomaly_scores[valid.to_numpy()]
            row["n_positive_test"] = int((y == 1).sum())
            row["positive_rate_test"] = (
                float((y == 1).mean()) if len(y) else np.nan
            )
            if len(y) > 0 and len(np.unique(y)) == 2:
                row["PR_AUC"] = average_precision_score(
                    y, scores_valid
                )
                row["ROC_AUC"] = roc_auc_score(
                    y, scores_valid
                )

                k = max(
                    1,
                    int(np.ceil(0.01 * len(scores_valid)))
                )

                precision, recall = precision_recall_at_k(
                    y, scores_valid, k
                )

                row["Precision_at_1pct"] = precision
                row["Recall_at_1pct"] = recall

        score_table[f"score_{name}"] = anomaly_scores

        rows.append(row)
        fitted[name] = {
            "model": model,
            "imputer": imputer,
            "features": usable_cols
        }

    return pd.DataFrame(rows), score_table, fitted


# Jalankan ulang seluruh eksperimen
ibm_results, ibm_scores, ibm_models = run_experiments(
    ibm_train,
    ibm_test,
    IBM_FEATURE_SETS,
    "IBM_AML"
)

paysim_results, paysim_scores, paysim_models = run_experiments(
    paysim_train,
    paysim_test,
    PAYSIM_FEATURE_SETS,
    "PaySim"
)

all_results = pd.concat(
    [ibm_results, paysim_results],
    ignore_index=True
)

display(all_results)


## 12. Diagnostics and plots

In [ ]:
for name, test in [("IBM AML", ibm_test), ("PaySim", paysim_test)]:
    print("\\n", name)
    if "label" in test.columns:
        labels = pd.to_numeric(test["label"], errors="coerce")
        display(labels.value_counts(dropna=False).to_frame("count"))
    else:
        print("No label mapped")

plot_data = all_results.dropna(subset=["PR_AUC"])
if not plot_data.empty:
    for ds, group in plot_data.groupby("dataset"):
        plt.figure(figsize=(8, 4))
        plt.bar(group.experiment, group.PR_AUC)
        plt.ylabel("PR-AUC"); plt.xlabel("Experiment")
        plt.title(f"PR-AUC by feature set — {ds}")
        plt.xticks(rotation=20); plt.tight_layout(); plt.show()
else:
    print("No PR-AUC available; verify label mapping and test-set class distribution")

plt.figure(figsize=(9, 4))
for col in [c for c in ibm_scores.columns if c.startswith("score_")]:
    plt.hist(ibm_scores[col], bins=50, alpha=0.35, label=col)
plt.xlabel("Anomaly score (higher = more anomalous)")
plt.ylabel("Count"); plt.title("IBM AML test anomaly-score distributions")
plt.legend(); plt.tight_layout(); plt.show()

## 13. Save results

In [ ]:

from pathlib import Path

output_dir = Path("fraud_detection_results")
output_dir.mkdir(parents=True, exist_ok=True)

# Simpan metrik evaluasi
ibm_results.to_csv(
    output_dir / "experiment_metrics_ibm_aml.csv",
    index=False
)

paysim_results.to_csv(
    output_dir / "experiment_metrics_paysim.csv",
    index=False
)

all_results.to_csv(
    output_dir / "experiment_metrics_all.csv",
    index=False
)

# Simpan anomaly scores untuk analisis lanjutan
ibm_scores.to_csv(
    output_dir / "ibm_aml_test_anomaly_scores.csv",
    index=False
)

paysim_scores.to_csv(
    output_dir / "paysim_test_anomaly_scores.csv",
    index=False
)

print("Hasil terbaru berhasil disimpan di:", output_dir.resolve())
display(all_results)


In [ ]:
from pathlib import Path
import shutil

output_dir = Path("fraud_detection_results")
if not output_dir.exists():
    raise FileNotFoundError(
        f"Folder hasil tidak ditemukan: {output_dir.resolve()}"
    )

zip_path = shutil.make_archive(
    str(output_dir.resolve()), "zip", root_dir=str(output_dir.resolve())
)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Bukan Google Colab", zip_path)


## 15. Model kandidat operasional: train–calibration–test dan threshold alert

Membangun kandidat terpisah berbasis E2 Temporal. Urutan waktunya dibagi menjadi:
- **60% awal:** melatih Isolation Forest.
- **20% berikutnya:** menentukan threshold alert dari skor anomali, tanpa melihat label.
- **20% terakhir:** evaluasi akhir yang tidak digunakan untuk memilih fitur atau threshold.

Skor anomali bukan probabilitas fraud. Threshold persentil mengatur kapasitas antrean alert, bukan membuktikan transaksi benar-benar fraud. Label hanya digunakan setelah threshold dibekukan untuk evaluasi.


In [ ]:

import json, joblib
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np
import pandas as pd

DEPLOY_DIR = Path("fraud_detection_candidate")
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

def chronological_three_way_split(feat, train_fraction=0.60, calibration_fraction=0.20):
    """Split kronologis tanpa memecah timestamp yang sama."""
    clean = feat.dropna(subset=["timestamp"]).copy()
    counts = clean.groupby("timestamp", sort=True).size()
    if len(counts) < 3:
        raise ValueError("Perlu setidaknya tiga timestamp unik untuk split tiga bagian")
    cumulative = counts.cumsum()
    n = len(clean)
    target_train = n * train_fraction
    train_pos = int(np.argmin(np.abs(cumulative.iloc[:-2].to_numpy() - target_train)))
    train_cut = counts.index[train_pos + 1]
    remaining = clean[clean["timestamp"] >= train_cut]
    rem_counts = remaining.groupby("timestamp", sort=True).size()
    target_cal = len(remaining) * (calibration_fraction / (1.0 - train_fraction))
    cal_pos = int(np.argmin(np.abs(rem_counts.cumsum().iloc[:-1].to_numpy() - target_cal)))
    cal_cut = rem_counts.index[cal_pos + 1]
    train = clean[clean["timestamp"] < train_cut].copy()
    calibration = clean[(clean["timestamp"] >= train_cut) & (clean["timestamp"] < cal_cut)].copy()
    test = clean[clean["timestamp"] >= cal_cut].copy()
    if min(len(train), len(calibration), len(test)) == 0:
        raise ValueError("Split train/calibration/test menghasilkan bagian kosong")
    assert train.timestamp.max() < calibration.timestamp.min()
    assert calibration.timestamp.max() < test.timestamp.min()
    return train, calibration, test, train_cut, cal_cut

def operational_features(frame, dataset_name):
    """Transformasi deterministik; tidak mempelajari parameter dari test"""
    x = pd.DataFrame(index=frame.index)
    x["amount_log"] = pd.to_numeric(frame["amount_log"], errors="coerce")
    x["source_prior_count_log"] = np.log1p(
        pd.to_numeric(frame["source_prior_count"], errors="coerce").clip(lower=0)
    )
    x["source_gap_seconds_log"] = np.log1p(
        pd.to_numeric(frame["source_seconds_since_previous"], errors="coerce").clip(lower=0)
    )
    if dataset_name == "IBM_AML":
        ts = pd.to_datetime(frame["timestamp"], errors="coerce", utc=True)
        hour = ts.dt.hour + ts.dt.minute / 60.0
        dow = ts.dt.dayofweek
        x["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
        x["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
        x["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
        x["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)
    return x.replace([np.inf, -np.inf], np.nan)

def evaluate_operational_model(dataset_name, feat, target_alert_rate=0.01, seed=42):
    train, calibration, test, train_cut, cal_cut = chronological_three_way_split(feat)
    X_train = operational_features(train, dataset_name)
    X_cal = operational_features(calibration, dataset_name)
    X_test = operational_features(test, dataset_name)

    usable = [c for c in X_train.columns if X_train[c].notna().any()]
    X_train, X_cal, X_test = X_train[usable], X_cal[usable], X_test[usable]
    imputer = SimpleImputer(strategy="median")
    Xtr = imputer.fit_transform(X_train)
    Xca = imputer.transform(X_cal)
    Xte = imputer.transform(X_test)

    model = IsolationForest(
        n_estimators=400,
        max_samples="auto",
        contamination="auto",
        random_state=seed,
        n_jobs=-1
    )
    model.fit(Xtr)

    # Skor lebih tinggi = lebih anomali
    cal_scores = -model.decision_function(Xca)
    test_scores = -model.decision_function(Xte)

    # Threshold dibekukan dari calibration scores, tanpa memakai label
    threshold = float(np.quantile(cal_scores, 1.0 - target_alert_rate))
    alerts = (test_scores >= threshold).astype(int)

    labels = pd.to_numeric(test.get("label", pd.Series(index=test.index, dtype=float)),
                           errors="coerce")
    valid = labels.notna()
    y = labels.loc[valid].astype(int).to_numpy()
    s = test_scores[valid.to_numpy()]
    a = alerts[valid.to_numpy()]

    metrics = {
        "dataset": dataset_name,
        "model": "IsolationForest_E2_temporal",
        "n_train": int(len(train)),
        "n_calibration": int(len(calibration)),
        "n_test": int(len(test)),
        "train_end": str(train["timestamp"].max()),
        "calibration_end": str(calibration["timestamp"].max()),
        "test_start": str(test["timestamp"].min()),
        "target_alert_rate": float(target_alert_rate),
        "threshold_from_calibration": threshold,
        "observed_test_alert_rate": float(alerts.mean()),
        "n_test_alerts": int(alerts.sum()),
        "n_positive_test": int((y == 1).sum()) if len(y) else None,
        "precision_at_threshold": float(y[a == 1].mean()) if len(y) and a.sum() else 0.0,
        "recall_at_threshold": float(((a == 1) & (y == 1)).sum() / (y == 1).sum())
            if len(y) and (y == 1).sum() else None,
        "average_precision": float(average_precision_score(y, s))
            if len(y) and len(np.unique(y)) == 2 else None,
        "roc_auc": float(roc_auc_score(y, s))
            if len(y) and len(np.unique(y)) == 2 else None,
        "features": usable,
        "warning": "Anomaly score is not a fraud probability; benchmark results are not production validation."
    }

    score_cols = [c for c in [
        "row_id", "timestamp", "source_account", "destination_account", "amount", "label"
    ] if c in test.columns]
    queue = test[score_cols].copy()
    queue["anomaly_score"] = test_scores
    queue["threshold"] = threshold
    queue["alert"] = alerts
    queue["priority_rank"] = queue["anomaly_score"].rank(method="first", ascending=False).astype(int)
    queue = queue.sort_values(["alert", "anomaly_score"], ascending=[False, False])

    safe_name = dataset_name.lower().replace(" ", "_")
    queue.to_csv(DEPLOY_DIR / f"{safe_name}_test_alert_queue.csv", index=False)
    joblib.dump(
        {"model": model, "imputer": imputer, "features": usable,
         "threshold": threshold, "target_alert_rate": target_alert_rate,
         "dataset": dataset_name, "score_direction": "higher_is_more_anomalous"},
        DEPLOY_DIR / f"{safe_name}_isolation_forest.joblib"
    )
    with open(DEPLOY_DIR / f"{safe_name}_model_card.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    return metrics, queue

operational_results = []
operational_queues = {}
for dataset_name, features in [("IBM_AML", ibm_feat), ("PaySim", paysim_feat)]:
    metrics, queue = evaluate_operational_model(
        dataset_name, features, target_alert_rate=0.01, seed=SEED
    )
    operational_results.append(metrics)
    operational_queues[dataset_name] = queue

operational_metrics = pd.DataFrame(operational_results)
operational_metrics.to_csv(DEPLOY_DIR / "operational_candidate_metrics.csv", index=False)
display(operational_metrics)
print("Artefak kandidat tersimpan di:", DEPLOY_DIR.resolve())
print("Peringatan: jangan deploy ke keputusan transaksi otomatis sebelum validasi pada data nyata.")


## 16. Ekspor seluruh hasil eksperimen dan kandidat operasional


In [ ]:

from pathlib import Path
import zipfile
import json

RESULTS_DIR = Path("fraud_detection_results")
CANDIDATE_DIR = Path("fraud_detection_candidate")
EXPORT_ZIP = Path("fraud_detection_results_complete.zip")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if CANDIDATE_DIR.exists():
    for source_file in CANDIDATE_DIR.iterdir():
        if source_file.is_file():
            target_file = RESULTS_DIR / source_file.name
            target_file.write_bytes(source_file.read_bytes())

with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for base in [RESULTS_DIR, CANDIDATE_DIR]:
        if base.exists():
            for file_path in base.rglob("*"):
                if file_path.is_file():
                    arcname = str(Path(base.name) / file_path.relative_to(base))
                    zf.write(file_path, arcname=arcname)

print("ZIP lengkap tersimpan di:", EXPORT_ZIP.resolve())
with zipfile.ZipFile(EXPORT_ZIP, "r") as zf:
    print("Isi ZIP:")
    for name in zf.namelist():
        print(" -", name)


## 17. Integrasi E1–E4 dan perbandingan model (satu split kronologis)

Menjalankan **E1–E4 secara konsisten** pada IBM AML dan PaySim, memakai batas train/validation/calibration/test yang sama per dataset. Eksperimen utama adalah ablation Isolation Forest. Label tidak pernah menjadi fitur dan tidak digunakan untuk fitting. Validation AP untuk pemilihan model. LOF dibatasi pada maksimum 15.000 baris train agar runtime/memori Colab lebih terkendali.

Definisi fitur mengikuti fungsi `feature_sets`.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from pathlib import Path
import numpy as np, pandas as pd, json, zipfile, joblib, shutil

INTEGRATED_DIR = Path("fraud_detection_e1_e4_integrated")
INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)
LOF_TRAIN_CAP = 15000
ALERT_RATE = 0.01
INTEGRATED_TOPK_RATES = [0.005, 0.01, 0.02, 0.05]

required_objects = ["ibm_feat", "paysim_feat", "feature_sets", "SEED"]
missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise RuntimeError(f"Objek pipeline belum tersedia: {missing_objects}. Jalankan notebook dari awal")

def split_four_way(frame):
    frame = frame.dropna(subset=["timestamp"]).sort_values(["timestamp"] + (["row_id"] if "row_id" in frame else [])).copy()
    counts = frame.groupby("timestamp", sort=True).size()
    if len(counts) < 4:
        raise ValueError("Tidak cukup timestamp unik untuk split kronologis")
    def split_at_fraction(left_frame, fraction):
        cnt = left_frame.groupby("timestamp", sort=True).size()
        cum = cnt.cumsum().iloc[:-1]
        if len(cum) == 0:
            raise ValueError("Tidak dapat membuat batas split")
        pos = int(np.argmin(np.abs(cum.to_numpy() - fraction * len(left_frame))))
        cut = cnt.index[pos + 1]
        return left_frame[left_frame.timestamp < cut].copy(), left_frame[left_frame.timestamp >= cut].copy(), cut
    train, rem, c1 = split_at_fraction(frame, .50)
    val, rem, c2 = split_at_fraction(rem, .40)       # about 20% of full dataset
    cal, test, c3 = split_at_fraction(rem, 1/3)      # about 10% + 20%
    assert train.timestamp.max() < val.timestamp.min() < cal.timestamp.min() < test.timestamp.min()
    return {"train":train,"validation":val,"calibration":cal,"test":test}, [c1,c2,c3]

def binary_labels(frame):
    if "label" not in frame.columns:
        raise ValueError("Kolom label tidak tersedia; evaluasi tidak dapat dilakukan")
    s = pd.to_numeric(frame["label"], errors="coerce")
    valid = s.notna().to_numpy()
    y = s.loc[s.notna()].astype(int).to_numpy()
    if not set(np.unique(y)).issubset({0,1}):
        raise ValueError(f"Label harus biner 0/1; ditemukan {np.unique(y)}")
    return y, valid

def eval_ranking(y, scores, prefix):
    y=np.asarray(y,dtype=int); scores=np.asarray(scores,dtype=float)
    finite=np.isfinite(scores); y=y[finite]; scores=scores[finite]
    out={}
    if len(y) and len(np.unique(y)) == 2:
        out[prefix+"average_precision"] = float(average_precision_score(y,scores))
        out[prefix+"roc_auc"] = float(roc_auc_score(y,scores))
    else:
        out[prefix+"average_precision"] = np.nan; out[prefix+"roc_auc"] = np.nan
    for rate in INTEGRATED_TOPK_RATES:
        k=max(1,int(np.ceil(rate*len(scores)))) if len(scores) else 0
        idx=np.argsort(-scores,kind="mergesort")[:k] if k else np.array([],dtype=int)
        tp=int(y[idx].sum()) if k else 0
        positives=int(y.sum())
        tag=f"{rate*100:g}pct"
        out[prefix+f"precision_at_{tag}"]=tp/k if k else np.nan
        out[prefix+f"recall_at_{tag}"]=tp/positives if positives else np.nan
        out[prefix+f"n_top_{tag}"]=k
    return out

def clean_feature_list(frame, requested):
    blocked={"label","is_fraud","isfraud","fraud","is_laundering","is_sar","timestamp","row_id","source_row_id",
             "source_account","destination_account","source_bank","destination_bank","time_index","amount"}
    cols=[c for c in requested if c in frame.columns and str(c).lower() not in blocked]
    cols=[c for c in cols if pd.api.types.is_numeric_dtype(frame[c])]
    return cols

all_metrics=[]; all_topk=[]; all_queues=[]; split_rows=[]
feature_definitions={}
for dataset_name, frame in [("IBM_AML", ibm_feat), ("PaySim", paysim_feat)]:
    frame=frame.copy()
    split, cuts=split_four_way(frame)
    fsets=feature_sets(frame)
    feature_definitions[dataset_name]={}
    for exp_name, requested in fsets.items():
        cols=clean_feature_list(frame, requested)
        if not cols:
            raise ValueError(f"{dataset_name}/{exp_name}: tidak ada fitur numerik yang dapat dipakai, requested={requested}")
        feature_definitions[dataset_name][exp_name]=cols
        parts={k:v.copy() for k,v in split.items()}
        X={k: v[cols].replace([np.inf,-np.inf],np.nan) for k,v in parts.items()}
        usable=[c for c in cols if X["train"][c].notna().any()]
        if not usable:
            raise ValueError(f"{dataset_name}/{exp_name}: seluruh fitur train kosong")
        X={k:v[usable] for k,v in X.items()}
        imputer=SimpleImputer(strategy="median")
        Xi={"train":imputer.fit_transform(X["train"])}
        for part in ["validation","calibration","test"]: Xi[part]=imputer.transform(X[part])
        scaler=StandardScaler()
        Xs={"train":scaler.fit_transform(Xi["train"])}
        for part in ["validation","calibration","test"]: Xs[part]=scaler.transform(Xi[part])
        scores={}; model_objects={}; model_meta={}
        # Primary model for E1-E4 ablation.
        model=IsolationForest(n_estimators=300,max_samples="auto",contamination="auto",random_state=SEED,n_jobs=-1)
        model.fit(Xi["train"])
        model_objects["IsolationForest"]=(model,"imputed")
        model_meta["IsolationForest"]={"n_estimators":300,"random_state":int(SEED)}
        scores["IsolationForest"]={p:-model.decision_function(Xi[p]) for p in ["validation","calibration","test"]}
        # Secondary model comparison on each identical feature set and split.
        n_fit=min(LOF_TRAIN_CAP,len(Xs["train"]))
        rng=np.random.default_rng(SEED)
        fit_idx=np.sort(rng.choice(len(Xs["train"]),size=n_fit,replace=False))
        lof=LocalOutlierFactor(n_neighbors=min(35,max(2,n_fit-1)),novelty=True,n_jobs=-1)
        lof.fit(Xs["train"][fit_idx])
        model_objects["LOF_novelty"]=(lof,"scaled")
        model_meta["LOF_novelty"]={"fit_rows":int(n_fit),"n_neighbors":int(min(35,max(2,n_fit-1)))}
        scores["LOF_novelty"]={p:-lof.score_samples(Xs[p]) for p in ["validation","calibration","test"]}
        n_features=Xs["train"].shape[1]
        n_components=max(1,min(3,n_features-1)) if n_features>1 else 1
        pca=PCA(n_components=n_components,random_state=SEED).fit(Xs["train"])
        model_objects["PCA_reconstruction"]=(pca,"scaled")
        model_meta["PCA_reconstruction"]={"n_components":int(n_components)}
        scores["PCA_reconstruction"]={p:np.mean((Xs[p]-pca.inverse_transform(pca.transform(Xs[p])))**2,axis=1) for p in ["validation","calibration","test"]}

        yval, vvalid=binary_labels(parts["validation"])
        ytest, tvalid=binary_labels(parts["test"])
        exp_rows=[]
        for model_name, sp in scores.items():
            row={"dataset":dataset_name,"experiment":exp_name,"model":model_name,
                 "features":";".join(usable),"n_features":len(usable),
                 "n_train":len(parts["train"]),"n_validation":len(parts["validation"]),
                 "n_calibration":len(parts["calibration"]),"n_test":len(parts["test"]),
                 "train_end":str(parts["train"].timestamp.max()),"validation_end":str(parts["validation"].timestamp.max()),
                 "calibration_end":str(parts["calibration"].timestamp.max()),"test_start":str(parts["test"].timestamp.min())}
            row.update(eval_ranking(yval,sp["validation"][vvalid],"validation_"))
            threshold=float(np.quantile(sp["calibration"],1-ALERT_RATE))
            alerts=(sp["test"]>=threshold)[tvalid]
            tp=int(((ytest==1)&alerts).sum()); n_alerts=int(alerts.sum()); n_pos=int((ytest==1).sum())
            row.update({"threshold_from_calibration":threshold,"test_alert_rate_all_rows":float((sp["test"]>=threshold).mean()),
                        "n_test_alerts_valid_label":n_alerts,"n_positive_test":n_pos,
                        "precision_at_threshold":tp/n_alerts if n_alerts else 0.0,
                        "recall_at_threshold":tp/n_pos if n_pos else np.nan})
            row.update(eval_ranking(ytest,sp["test"][tvalid],"test_"))
            exp_rows.append(row)
            # Save score queue for reproducibility/inspection
            keep=[c for c in ["row_id","timestamp","source_account","destination_account","amount","label"] if c in parts["test"].columns]
            q=parts["test"][keep].copy(); q["dataset"]=dataset_name; q["experiment"]=exp_name; q["model"]=model_name
            q["anomaly_score"]=sp["test"]; q["threshold_from_calibration"]=threshold
            q["alert"]=(q["anomaly_score"]>=threshold).astype(int)
            q["priority_rank"]=q["anomaly_score"].rank(method="first",ascending=False).astype(int)
            all_queues.append(q)
            # top-k capacity table
            valid_scores=sp["test"][tvalid]
            for rate in INTEGRATED_TOPK_RATES:
                k=max(1,int(np.ceil(rate*len(valid_scores))))
                idx=np.argsort(-valid_scores,kind="mergesort")[:k]
                tp_k=int(ytest[idx].sum()); npos=int((ytest==1).sum())
                all_topk.append({"dataset":dataset_name,"experiment":exp_name,"model":model_name,
                                 "capacity_rate":rate,"n_alerts":k,"true_positives":tp_k,
                                 "precision_at_k":tp_k/k,"recall_at_k":tp_k/npos if npos else np.nan})
        all_metrics.extend(exp_rows)
        # Selection is strictly validation AP, within each dataset/experiment
        exp_df=pd.DataFrame(exp_rows).sort_values(["validation_average_precision","validation_recall_at_1pct"],ascending=False,na_position="last")
        winner=exp_df.iloc[0]
        chosen_model_name=str(winner["model"])
        chosen_model,space=model_objects[chosen_model_name]
        bundle={"dataset":dataset_name,"experiment":exp_name,"selected_model_by_validation_ap":chosen_model_name,
                "model":chosen_model,"imputer":imputer,"scaler":scaler if space=="scaled" else None,
                "features":usable,"threshold":float(np.quantile(scores[chosen_model_name]["calibration"],1-ALERT_RATE)),
                "target_alert_rate":ALERT_RATE,"model_metadata":model_meta[chosen_model_name],
                "warning":"Research benchmark only; anomaly scores are not calibrated fraud probabilities."}
        joblib.dump(bundle,INTEGRATED_DIR/f"{dataset_name.lower()}_{exp_name.lower()}_{chosen_model_name.lower()}.joblib")
        split_rows.extend([{"dataset":dataset_name,"split":part,"rows":len(d),"start":str(d.timestamp.min()),"end":str(d.timestamp.max()),
                           "positive_labels":int(pd.to_numeric(d.label,errors="coerce").eq(1).sum()) if "label" in d else None}
                          for part,d in parts.items()])
        print(f"{dataset_name} | {exp_name}: validation-AP winner = {chosen_model_name}; features={usable}")

metrics_df=pd.DataFrame(all_metrics)
topk_df=pd.DataFrame(all_topk)
queues_df=pd.concat(all_queues,ignore_index=True)
splits_df=pd.DataFrame(split_rows)
metrics_df.to_csv(INTEGRATED_DIR/"e1_e4_model_metrics.csv",index=False)
topk_df.to_csv(INTEGRATED_DIR/"e1_e4_topk_capacity.csv",index=False)
queues_df.to_csv(INTEGRATED_DIR/"e1_e4_test_scores_and_alert_queues.csv",index=False)
splits_df.to_csv(INTEGRATED_DIR/"chronological_split_summary.csv",index=False)
with open(INTEGRATED_DIR/"feature_definitions.json","w",encoding="utf-8") as f: json.dump(feature_definitions,f,indent=2,ensure_ascii=False)

if_ablation=metrics_df[metrics_df.model.eq("IsolationForest")].copy()
if_ablation.to_csv(INTEGRATED_DIR/"primary_isolation_forest_e1_e4_ablation.csv",index=False)
winners=(metrics_df.sort_values(["dataset","experiment","validation_average_precision","validation_recall_at_1pct"],ascending=[True,True,False,False],na_position="last")
          .groupby(["dataset","experiment"],as_index=False).first())
winners.to_csv(INTEGRATED_DIR/"validation_selected_model_per_experiment.csv",index=False)
print("\nE1–E4 Isolation Forest ablation (primary):")
display(if_ablation[["dataset","experiment","n_features","validation_average_precision","test_average_precision","test_roc_auc","test_recall_at_0.5pct","test_recall_at_1pct","test_recall_at_2pct","test_recall_at_5pct"]])
print("\nAll models across E1–E4:")
display(metrics_df.sort_values(["dataset","experiment","validation_average_precision"],ascending=[True,True,False]))
print("\nTop-k capacity results:")
display(topk_df)
print("\nChronological split summary:")
display(splits_df)


## 18. Export hasil integrasi E1–E4

In [ ]:
from pathlib import Path
import zipfile

FINAL_ZIP = Path("fraud_detection_e1_e4_integrated_complete.zip")
include_dirs = [Path("fraud_detection_results"), Path("fraud_detection_candidate"), INTEGRATED_DIR]
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for base_dir in include_dirs:
        if base_dir.exists():
            for file_path in base_dir.rglob("*"):
                if file_path.is_file():
                    zf.write(file_path, arcname=str(Path(base_dir.name) / file_path.relative_to(base_dir)))
print("Paket terintegrasi:", FINAL_ZIP.resolve())
with zipfile.ZipFile(FINAL_ZIP) as zf:
    print("Jumlah file:", len(zf.namelist()))
    for name in zf.namelist(): print(" -", name)


## 19. Robustness evaluation: expanding-window chronological holdouts

**Interpretation:** folds with no positive labels cannot yield meaningful AP/ROC-AUC and are recorded as unavailable. If too few folds contain positive examples, increase the sample size or use the full source feature table before drawing conclusions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from pathlib import Path

ROLLING_DIR = Path("fraud_detection_rolling_evaluation")
ROLLING_DIR.mkdir(parents=True, exist_ok=True)

ROLLING_TRAIN_FRACTION = 0.50
ROLLING_TEST_FRACTION = 0.10
ROLLING_MAX_FOLDS = 5
ROLLING_TOPK_RATES = [0.005, 0.01, 0.02, 0.05]
ROLLING_N_ESTIMATORS = 300
ROLLING_SEED = int(globals().get("SEED", 42))

def make_expanding_time_folds(frame, train_fraction=0.50, test_fraction=0.10,
                              max_folds=5):
    """Create forward-only folds using unique timestamps; never split a timestamp"""
    if "timestamp" not in frame.columns:
        raise KeyError("Kolom timestamp tidak ditemukan")
    data = frame.dropna(subset=["timestamp"]).copy()
    data = data.sort_values(["timestamp"] + (["row_id"] if "row_id" in data.columns else []))
    time_counts = data.groupby("timestamp", sort=True).size()
    times = time_counts.index.to_list()
    n_times = len(times)
    if n_times < 10:
        raise ValueError("Timestamp unik terlalu sedikit untuk rolling evaluation")
    first_train_end = max(1, int(np.floor(n_times * train_fraction)))
    test_width = max(1, int(np.floor(n_times * test_fraction)))
    folds = []
    start = first_train_end
    fold_id = 1
    while start < n_times and fold_id <= max_folds:
        stop = min(start + test_width, n_times)
        train_end_time = times[start - 1]
        test_start_time = times[start]
        test_end_time = times[stop - 1]
        train = data[data["timestamp"] <= train_end_time].copy()
        test = data[(data["timestamp"] >= test_start_time) &
                    (data["timestamp"] <= test_end_time)].copy()
        if len(train) and len(test):
            assert train["timestamp"].max() < test["timestamp"].min()
            folds.append((fold_id, train, test))
        start = stop
        fold_id += 1
    return folds

def safe_binary_labels(frame):
    if "label" not in frame.columns:
        return None
    labels = pd.to_numeric(frame["label"], errors="coerce")
    valid = labels.notna()
    y = labels.loc[valid].astype(int).to_numpy()
    if not set(np.unique(y)).issubset({0, 1}):
        raise ValueError(f"Label harus biner 0/1; ditemukan {np.unique(y)}")
    return y, valid.to_numpy()

def evaluate_rolling_fold(train, test, feature_cols, dataset_name, experiment, fold_id):
    blocked = {"label", "is_fraud", "isfraud", "fraud", "is_laundering", "is_sar",
               "timestamp", "row_id", "source_row_id", "source_account",
               "destination_account", "source_bank", "destination_bank",
               "time_index", "amount", "isflaggedfraud"}
    forbidden = [c for c in feature_cols if str(c).lower() in blocked]
    if forbidden:
        raise AssertionError(f"Potential leakage features: {forbidden}")

    X_train = train[feature_cols].apply(pd.to_numeric, errors="coerce")
    X_test = test[feature_cols].apply(pd.to_numeric, errors="coerce")
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    usable = [c for c in feature_cols if X_train[c].notna().any()]
    if not usable:
        raise ValueError(f"{dataset_name}/{experiment}/fold {fold_id}: no usable train features")
    X_train, X_test = X_train[usable], X_test[usable]

    imputer = SimpleImputer(strategy="median")
    Xtr = imputer.fit_transform(X_train)
    Xte = imputer.transform(X_test)
    model = IsolationForest(
        n_estimators=ROLLING_N_ESTIMATORS,
        max_samples="auto",
        contamination="auto",
        random_state=ROLLING_SEED,
        n_jobs=-1
    )
    model.fit(Xtr)
    scores = -model.decision_function(Xte)  # larger = more anomalous

    labels = pd.to_numeric(test.get("label", pd.Series(index=test.index, dtype=float)),
                           errors="coerce")
    valid = labels.notna().to_numpy()
    y = labels.loc[labels.notna()].astype(int).to_numpy()
    s = scores[valid]
    positives = int((y == 1).sum()) if len(y) else 0
    result = {
        "dataset": dataset_name,
        "experiment": experiment,
        "fold": fold_id,
        "train_start": str(train["timestamp"].min()),
        "train_end": str(train["timestamp"].max()),
        "test_start": str(test["timestamp"].min()),
        "test_end": str(test["timestamp"].max()),
        "n_train": int(len(train)),
        "n_test": int(len(test)),
        "n_test_labels_valid": int(len(y)),
        "n_positive_test": positives,
        "positive_rate_test": float((y == 1).mean()) if len(y) else np.nan,
        "n_features_used": int(len(usable)),
        "average_precision": np.nan,
        "roc_auc": np.nan
    }
    if len(y) and len(np.unique(y)) == 2:
        result["average_precision"] = float(average_precision_score(y, s))
        result["roc_auc"] = float(roc_auc_score(y, s))
    for rate in ROLLING_TOPK_RATES:
        k = min(len(s), max(1, int(np.ceil(rate * len(s))))) if len(s) else 0
        tag = f"{rate*100:g}pct"
        if k:
            top_idx = np.argsort(-s, kind="mergesort")[:k]
            tp = int(y[top_idx].sum())
            result[f"precision_at_{tag}"] = tp / k
            result[f"recall_at_{tag}"] = tp / positives if positives else np.nan
            result[f"n_top_{tag}"] = k
        else:
            result[f"precision_at_{tag}"] = np.nan
            result[f"recall_at_{tag}"] = np.nan
            result[f"n_top_{tag}"] = 0
    return result

rolling_rows = []
for dataset_name, frame in [("IBM_AML", ibm_feat), ("PaySim", paysim_feat)]:
    print(f"\nRolling evaluation: {dataset_name}")
    sets = feature_sets(frame)
    folds = make_expanding_time_folds(
        frame,
        train_fraction=ROLLING_TRAIN_FRACTION,
        test_fraction=ROLLING_TEST_FRACTION,
        max_folds=ROLLING_MAX_FOLDS
    )
    print(f"Jumlah fold kronologis: {len(folds)}")
    for fold_id, train_fold, test_fold in folds:
        print(f" Fold {fold_id}: train={len(train_fold):,}, test={len(test_fold):,}, "
              f"positives={pd.to_numeric(test_fold['label'], errors='coerce').eq(1).sum() if 'label' in test_fold else 'N/A'}")
        for experiment, feature_cols in sets.items():
            rolling_rows.append(
                evaluate_rolling_fold(
                    train_fold, test_fold, feature_cols,
                    dataset_name, experiment, fold_id
                )
            )

rolling_results = pd.DataFrame(rolling_rows)
rolling_csv = ROLLING_DIR / "rolling_window_metrics.csv"
rolling_results.to_csv(rolling_csv, index=False)
display(rolling_results)
print("Rolling-window metrics saved to:", rolling_csv.resolve())


## 20. Summarize robustness across chronological folds

Mean and standard deviation across folds, but also retain per-fold metrics and positive counts. Do not interpret an average as robust if most folds have no positive labels. A fold-level table should be retained in the research supplement.


In [ ]:
if "rolling_results" not in globals() or rolling_results.empty:
    raise RuntimeError("Jalankan section 19 terlebih dahulu.")

metric_cols = [
    "average_precision", "roc_auc",
    "precision_at_0.5pct", "recall_at_0.5pct",
    "precision_at_1pct", "recall_at_1pct",
    "precision_at_2pct", "recall_at_2pct",
    "precision_at_5pct", "recall_at_5pct"
]
available_metrics = [c for c in metric_cols if c in rolling_results.columns]
rolling_summary = (
    rolling_results
    .groupby(["dataset", "experiment"], dropna=False)[available_metrics]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary_path = ROLLING_DIR / "rolling_window_summary.csv"
rolling_summary.to_csv(summary_path, index=False)

# Plot AP per fold. NaN means that fold did not contain both label classes.
for dataset_name, group in rolling_results.groupby("dataset"):
    pivot = group.pivot(index="fold", columns="experiment", values="average_precision")
    ax = pivot.plot(kind="line", marker="o", figsize=(9, 4))
    ax.set_title(f"Average Precision by forward test window — {dataset_name}")
    ax.set_xlabel("Chronological fold")
    ax.set_ylabel("Average Precision")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

display(rolling_summary)
print("Summary saved to:", summary_path.resolve())
